<a href="https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/00_reserve_networking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Apigee X — Reserve Peering Ranges & Deploy Regional Subnet

This notebook prepares the **Google Cloud VPC networking prerequisites** for an **Apigee X** organization. It is **networking-only**: it stops before Apigee organization creation and outputs a clean handoff block for your Apigee X provisioning workflow.

> 🐙 **GitHub-Ready Execution (`luiscuellarh/apigee-unified-ai-gateway`):**
> 1. **Google Colab from GitHub (1-Click):** Click the **Open in Colab** button above, or open:
>    `https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/00_reserve_networking.ipynb`
> 2. **GitHub Codespaces:** Open this repository in a Codespace, authenticate once via `gcloud auth login --no-launch-browser` (or set `GOOGLE_APPLICATION_CREDENTIALS`), and run all cells.
> 3. **GitHub Actions (Headless CI/CD):** Execute via `jupyter nbconvert --to notebook --execute 00_reserve_networking.ipynb` with parameters passed as environment variables (`PROJECT_ID`, `REGION`, `NETWORK_NAME`, `DRY_RUN`, etc.).

---

## 🚨 CRITICAL CONCEPT — READ FIRST

This notebook performs **two fundamentally different networking operations**. Do not conflate them:

* **A) `RESERVE` (allocate only — nothing is deployed, no resources live here):**
  The Apigee runtime `/22` (or `/28`) range and the `/28` support range are **GLOBAL INTERNAL IP ALLOCATIONS** created with:
  `gcloud compute addresses create --global --purpose=VPC_PEERING`
  They are IP address reservations handed to Google's **Service Networking** (`servicenetworking.googleapis.com`) so the Google-managed Apigee tenant project can later consume them when provisioning runtime pods and internal load balancers.
  * They are **NOT** subnets.
  * They have **NO** region (they are global VPC-level address objects).
  * You must **NEVER** create a subnetwork over them.

* **B) `DEPLOY` (a real regional resource in your VPC):**
  **ONE regional subnetwork** in your selected `REGION`. This is the **only resource actually deployed** inside your VPC for your own customer-side workloads — Private Service Connect (PSC) NEGs, load-balancer bridges, Envoy/MIG proxies, test VMs, and private backends. Its CIDR is completely independent of, and must never overlap with, the reserved ranges in **(A)**.

```text
  [Your VPC] ── subnet (DEPLOYED, regional, your workloads)
         └──── reserved /22 + /28 (ALLOCATION ONLY)
                     │ servicenetworking peering
                     ▼
               [Apigee tenant project — Google-managed]
```

> ⚠️ **WARNING — Reserved Peering Ranges Cannot Be Resized Later:**
> Once your Apigee X organization is created against a reserved runtime peering range, that range **cannot be resized in-place**. Always allocate a **`/22`** (`1,024` IPs) for production or multi-environment organizations. A **`/28`** (`16` IPs) is strictly for minimal-footprint evaluation or sandbox orgs.

> ⚠️ **WARNING — Never Create a Subnetwork Inside a Reserved Peering Range:**
> Do **not** run `gcloud compute networks subnets create` using `PEERING_RANGE_BASE_IP` or `SUPPORT_RANGE_BASE_IP`. Creating a regional subnet inside a reserved range consumes the IP block locally in your VPC and prevents Service Networking / Apigee from peering with it.

---

### 📊 Apigee X Runtime Peering Range Sizing Comparison (`/22` vs `/28`)

| Dimension | `/22` Runtime Range (`1,024` IPs) | `/28` Runtime Range (`16` IPs) | `/28` Support Range (`16` IPs) |
| :--- | :--- | :--- | :--- |
| **Action Type** | `RESERVE (global)` | `RESERVE (global)` | `RESERVE (global)` |
| **Instance Sizes Supported** | Small, Medium, Large, Enterprise scale | Single small / evaluation instance only | N/A (Control-plane & support ops) |
| **Expandable / Multi-Instance** | ✅ Yes (supports multiple instances & environments) | ❌ No (hard cap on pod/ILB scaling) | Fixed `/28` per peering connection |
| **Multi-Region Headroom** | ✅ Yes (can share across multiple regions or add `/22`s) | ❌ No (single region, minimal capacity) | Fixed `/28` required alongside runtime |
| **Recommended Use** | **Production**, staging, and long-lived orgs | **Evaluation**, PoC, or short-lived dev/test | **Mandatory** for all peered Apigee X orgs |

> 💡 **Note on Private Service Connect (`disableVpcPeering=true`):**
> If you plan to create your Apigee X organization with **`disableVpcPeering=true`** (Private Service Connect / PSC-only networking without VPC Service Networking peering), **only the regional subnet in Step 4 is needed** — Steps 5 and 6 can be skipped (or set `CONNECT_PEERING = False`).

---

### ⚙️ Step 0 Overview — Authentication & Configuration Form
* **What it does:** Authenticates your session with Google Cloud (via `google.colab.auth` in Colab, or via Workload Identity / `GOOGLE_APPLICATION_CREDENTIALS` / ambient `gcloud` in GitHub Codespaces & GitHub Actions), captures all project, VPC, regional subnet (`DEPLOY`), and global IP range (`RESERVE`) parameters, defines the idempotent `run(cmd, description)` helper, exports variables to `os.environ`, and sets the active `gcloud` project.
* **Expected duration:** `~5–15 seconds` (plus interactive browser sign-in if not yet authenticated).
* **Plane:** `CONFIGURATION & AUTH` (prepares environment; neither reserves nor deploys resources).

In [ ]:
#@title ⚙️ Step 0: Authentication & Configuration Form { display-mode: "form" }

import ipaddress
import json
import os
import shlex
import subprocess
import sys

# Snapshot initial process environment once so headless GitHub Actions / Codespaces
# environment variables can populate defaults without overriding interactive Colab form edits.
if "_INITIAL_ENV" not in globals():
    _INITIAL_ENV = dict(os.environ)

#@markdown ---
#@markdown ### 1. Project & Region
PROJECT_ID = "" #@param {type:"string"}
REGION = "europe-west1" #@param ["europe-west1", "europe-west2", "europe-west3", "europe-west4", "europe-west6", "europe-southwest1", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-south1", "asia-southeast1", "australia-southeast1", "southamerica-east1"]

#@markdown ---
#@markdown ### 2. VPC Network
NETWORK_NAME = "apigee-network" #@param {type:"string"}
CREATE_NETWORK_IF_MISSING = True #@param {type:"boolean"}
#@markdown Created in CUSTOM subnet mode so no auto subnets collide
#@markdown with the reserved ranges.

#@markdown ---
#@markdown ### 3. Subnet to DEPLOY (regional, real resource)
SUBNET_NAME = "apigee-subnet" #@param {type:"string"}
SUBNET_CIDR = "10.10.0.0/24" #@param {type:"string"}
ENABLE_PRIVATE_GOOGLE_ACCESS = True #@param {type:"boolean"}
ENABLE_FLOW_LOGS = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 4. Ranges to RESERVE (global allocations, not deployed)
PEERING_RANGE_NAME = "apigee-peering-range" #@param {type:"string"}
PEERING_CIDR_SIZE = "/22" #@param ["/22", "/28"]
PEERING_RANGE_BASE_IP = "10.20.0.0" #@param {type:"string"}
SUPPORT_RANGE_NAME = "apigee-support-range" #@param {type:"string"}
SUPPORT_RANGE_BASE_IP = "10.30.0.0" #@param {type:"string"}
#@markdown The support range is ALWAYS /28, independent of the runtime size.
AUTO_ALLOCATE = False #@param {type:"boolean"}
#@markdown If True, omit --addresses and let Google choose the base IPs;
#@markdown only --prefix-length is passed.

#@markdown ---
#@markdown ### 5. Behaviour
CONNECT_PEERING = True #@param {type:"boolean"}
#@markdown If True, also run `gcloud services vpc-peerings connect` to
#@markdown hand the reserved ranges to Service Networking. If False, the
#@markdown ranges are reserved and left unattached for a later step.
DRY_RUN = False #@param {type:"boolean"}


def _env_str(key: str, current: str, default: str) -> str:
    """Uses startup environment variable in GitHub Actions/Codespaces if widget is at default."""
    if current != default:
        return current
    return _INITIAL_ENV.get(key, current)


def _env_bool(key: str, current: bool, default: bool) -> bool:
    """Uses startup boolean environment variable in GitHub Actions/Codespaces if widget is at default."""
    if current != default or key not in _INITIAL_ENV:
        return current
    return _INITIAL_ENV[key].strip().lower() in ("1", "true", "yes", "y")


# Allow seamless parameter injection when executing in GitHub Actions / GitHub Codespaces
PROJECT_ID = PROJECT_ID.strip() or _INITIAL_ENV.get("PROJECT_ID", "") or _INITIAL_ENV.get("GCP_PROJECT_ID", "")
REGION = _env_str("REGION", REGION, "europe-west1")
NETWORK_NAME = _env_str("NETWORK_NAME", NETWORK_NAME, "apigee-network")
CREATE_NETWORK_IF_MISSING = _env_bool("CREATE_NETWORK_IF_MISSING", CREATE_NETWORK_IF_MISSING, True)
SUBNET_NAME = _env_str("SUBNET_NAME", SUBNET_NAME, "apigee-subnet")
SUBNET_CIDR = _env_str("SUBNET_CIDR", SUBNET_CIDR, "10.10.0.0/24")
ENABLE_PRIVATE_GOOGLE_ACCESS = _env_bool("ENABLE_PRIVATE_GOOGLE_ACCESS", ENABLE_PRIVATE_GOOGLE_ACCESS, True)
ENABLE_FLOW_LOGS = _env_bool("ENABLE_FLOW_LOGS", ENABLE_FLOW_LOGS, True)
PEERING_RANGE_NAME = _env_str("PEERING_RANGE_NAME", PEERING_RANGE_NAME, "apigee-peering-range")
PEERING_CIDR_SIZE = _env_str("PEERING_CIDR_SIZE", PEERING_CIDR_SIZE, "/22")
PEERING_RANGE_BASE_IP = _env_str("PEERING_RANGE_BASE_IP", PEERING_RANGE_BASE_IP, "10.20.0.0")
SUPPORT_RANGE_NAME = _env_str("SUPPORT_RANGE_NAME", SUPPORT_RANGE_NAME, "apigee-support-range")
SUPPORT_RANGE_BASE_IP = _env_str("SUPPORT_RANGE_BASE_IP", SUPPORT_RANGE_BASE_IP, "10.30.0.0")
AUTO_ALLOCATE = _env_bool("AUTO_ALLOCATE", AUTO_ALLOCATE, False)
CONNECT_PEERING = _env_bool("CONNECT_PEERING", CONNECT_PEERING, True)
DRY_RUN = _env_bool("DRY_RUN", DRY_RUN, False)

APIGEE_SUPPORTED_REGIONS = (
    "europe-west1",
    "europe-west2",
    "europe-west3",
    "europe-west4",
    "europe-west6",
    "europe-southwest1",
    "us-central1",
    "us-east1",
    "us-east4",
    "us-west1",
    "asia-east1",
    "asia-northeast1",
    "asia-south1",
    "asia-southeast1",
    "australia-southeast1",
    "southamerica-east1",
)

# Global tracker for step outcomes and discovered/allocated CIDRs
STEP_RESULTS: dict[int, dict[str, str]] = {}
STATE: dict[str, str] = {
    "allocated_peering_cidr": (
        f"<auto-allocated>{PEERING_CIDR_SIZE}"
        if AUTO_ALLOCATE
        else f"{PEERING_RANGE_BASE_IP.strip()}{PEERING_CIDR_SIZE.strip()}"
    ),
    "allocated_support_cidr": (
        "<auto-allocated>/28"
        if AUTO_ALLOCATE
        else f"{SUPPORT_RANGE_BASE_IP.strip()}/28"
    ),
    "peering_state": "NOT_CONNECTED" if not CONNECT_PEERING else "PENDING",
    "subnet_usable_hosts": "N/A",
}


def record_step(step_num: int, name: str, status: str, detail: str = "") -> None:
    """Records step status and prints the mandatory standardized status line."""
    STEP_RESULTS[step_num] = {
        "step": f"Step {step_num}",
        "name": name,
        "status": status,
        "detail": detail,
    }
    if status == "COMPLETED":
        suffix = f" ({detail})" if detail else ""
        print(f"\n✅ Step {step_num} — {name} completed{suffix}")
    elif status == "SKIPPED":
        reason = detail or "already exists"
        print(f"\nℹ️ Step {step_num} — skipped ({reason}) [{name}]")
    elif status == "FAILED":
        reason = detail or "unknown error"
        print(f"\n❌ Step {step_num} — failed: {reason} [{name}]")


def run(cmd: str | list[str], description: str) -> str:
    """Executes a gcloud command while honouring DRY_RUN.

    Captures stdout/stderr and raises RuntimeError with a readable error message
    on non-zero exit codes. Never uses shell '|| true' suppression.
    """
    if isinstance(cmd, list):
        cmd_args = [str(part) for part in cmd]
        cmd_str = " ".join(shlex.quote(part) for part in cmd_args)
    else:
        cmd_str = " ".join(cmd.strip().split())
        cmd_args = shlex.split(cmd_str)

    print(f"▶️ {description}")
    print(f"   $ {cmd_str}")

    if DRY_RUN:
        print("   🧪 [DRY_RUN=True] Command printed only — execution skipped.")
        return ""

    proc = subprocess.run(
        cmd_args,
        capture_output=True,
        text=True,
        check=False,
        env=os.environ.copy(),
    )
    if proc.returncode != 0:
        err_msg = (proc.stderr or proc.stdout or "No error output captured").strip()
        raise RuntimeError(
            f"{description} failed (exit code {proc.returncode}).\n"
            f"  Command: {cmd_str}\n"
            f"  Details: {err_msg}"
        )
    return proc.stdout.strip()


try:
    # 1. Authenticate (supports Google Colab, GitHub Actions, GitHub Codespaces, and Service Account keys)
    try:
        from google.colab import auth  # type: ignore

        if not DRY_RUN:
            auth.authenticate_user()
            print("🔐 Authenticated user via google.colab.auth.")
        else:
            print("🧪 DRY_RUN=True: skipping interactive google.colab.auth prompt.")
    except ImportError:
        if os.environ.get("GITHUB_ACTIONS") == "true":
            print(
                f"🐙 Running inside GitHub Actions (repository: {os.environ.get('GITHUB_REPOSITORY', 'unknown')})."
            )
        elif os.environ.get("CODESPACES") == "true":
            print(
                f"🐙 Running inside GitHub Codespaces ({os.environ.get('CODESPACE_NAME', 'codespace')})."
            )
        else:
            print("ℹ️ google.colab not detected; using ambient gcloud / service-account credentials.")

        sa_key_file = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "").strip()
        if sa_key_file and os.path.isfile(sa_key_file) and not DRY_RUN:
            run(
                f"gcloud auth activate-service-account --key-file={shlex.quote(sa_key_file)}",
                "Activate service account from GOOGLE_APPLICATION_CREDENTIALS",
            )

    # 2. Export all configuration values to os.environ
    ENV_EXPORTS = {
        "PROJECT_ID": PROJECT_ID.strip(),
        "REGION": REGION.strip(),
        "NETWORK_NAME": NETWORK_NAME.strip(),
        "CREATE_NETWORK_IF_MISSING": str(CREATE_NETWORK_IF_MISSING),
        "SUBNET_NAME": SUBNET_NAME.strip(),
        "SUBNET_CIDR": SUBNET_CIDR.strip(),
        "ENABLE_PRIVATE_GOOGLE_ACCESS": str(ENABLE_PRIVATE_GOOGLE_ACCESS),
        "ENABLE_FLOW_LOGS": str(ENABLE_FLOW_LOGS),
        "PEERING_RANGE_NAME": PEERING_RANGE_NAME.strip(),
        "PEERING_CIDR_SIZE": PEERING_CIDR_SIZE.strip(),
        "PEERING_RANGE_BASE_IP": PEERING_RANGE_BASE_IP.strip(),
        "SUPPORT_RANGE_NAME": SUPPORT_RANGE_NAME.strip(),
        "SUPPORT_RANGE_BASE_IP": SUPPORT_RANGE_BASE_IP.strip(),
        "AUTO_ALLOCATE": str(AUTO_ALLOCATE),
        "CONNECT_PEERING": str(CONNECT_PEERING),
        "DRY_RUN": str(DRY_RUN),
    }
    for env_key, env_val in ENV_EXPORTS.items():
        os.environ[env_key] = env_val

    # 3. Configure active gcloud project
    if PROJECT_ID.strip():
        run(
            f"gcloud config set project {shlex.quote(PROJECT_ID.strip())}",
            f"Set active gcloud project to '{PROJECT_ID.strip()}'",
        )
    else:
        print("⚠️ PROJECT_ID is currently empty; Step 1 Pre-Flight Validation will fail fast unless set.")

    record_step(
        0,
        "Auth & Configuration Form",
        "COMPLETED",
        f"project='{PROJECT_ID.strip() or '<unset>'}', region='{REGION}', DRY_RUN={DRY_RUN}",
    )
except Exception as exc:
    record_step(0, "Auth & Configuration Form", "FAILED", str(exc))
    raise

### 🔍 Step 1: Pre-Flight Validation
* **What it does:** Validates all inputs and existing Google Cloud project resources **before** creating or modifying anything:
  1. Verifies `PROJECT_ID` is non-empty and accessible by the authenticated principal.
  2. Verifies `REGION` is in the official Apigee X supported regions list.
  3. Parses `SUBNET_CIDR` and both reserved-range base IPs (`PEERING_RANGE_BASE_IP` + `PEERING_CIDR_SIZE` and `SUPPORT_RANGE_BASE_IP` + `/28`) using Python's `ipaddress` stdlib module.
  4. Verifies all ranges fall strictly within **RFC1918 private IPv4 address space** (`10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16`).
  5. Verifies `SUBNET_CIDR` (`DEPLOY`) does **not** overlap the runtime peering range (`RESERVE`) or the support range (`RESERVE`), and that the two reserved ranges do not overlap each other.
  6. Queries existing subnets and existing global reserved IP ranges in `PROJECT_ID` to ensure zero overlap with existing project infrastructure.
  7. Renders a `✅` plan table with an explicit **`Type`** column reading **`DEPLOY (regional)`** vs **`RESERVE (global)`**.
* **Expected duration:** `~3–8 seconds`.
* **Plane:** `VALIDATION ONLY` (read-only checks; neither reserves nor deploys).

In [ ]:
#@title 🔍 Step 1: Pre-Flight Validation

RFC1918_BLOCKS = (
    ipaddress.IPv4Network("10.0.0.0/8"),
    ipaddress.IPv4Network("172.16.0.0/12"),
    ipaddress.IPv4Network("192.168.0.0/16"),
)


def is_rfc1918_network(net: ipaddress.IPv4Network) -> bool:
    """Returns True iff the IPv4Network is completely contained within RFC1918 private space."""
    return any(net.subnet_of(block) for block in RFC1918_BLOCKS)


def fail_preflight(reason: str) -> None:
    record_step(1, "Pre-Flight Validation", "FAILED", reason)
    raise ValueError(f"❌ Pre-Flight Validation Failed: {reason}")


try:
    # 1. Validate PROJECT_ID is non-empty and accessible
    proj_id = PROJECT_ID.strip()
    if not proj_id:
        fail_preflight("PROJECT_ID must not be empty. Please set PROJECT_ID in Step 0 and re-run.")

    proj_out = run(
        f"gcloud projects describe {shlex.quote(proj_id)} --format=json",
        f"Verify caller access to project '{proj_id}'",
    )
    if proj_out:
        proj_info = json.loads(proj_out)
        lifecycle = proj_info.get("lifecycleState", "UNKNOWN")
        if lifecycle != "ACTIVE":
            fail_preflight(f"Project '{proj_id}' is in lifecycleState '{lifecycle}' (expected 'ACTIVE').")

    # 2. Validate REGION is in the Apigee-supported list
    reg = REGION.strip()
    if reg not in APIGEE_SUPPORTED_REGIONS:
        fail_preflight(
            f"Region '{reg}' is not a supported Apigee X region. "
            f"Supported regions: {', '.join(APIGEE_SUPPORTED_REGIONS)}"
        )

    # 3. Parse SUBNET_CIDR and both reserved-range base IPs with `ipaddress`
    raw_subnet_cidr = SUBNET_CIDR.strip()
    if "/" not in raw_subnet_cidr:
        fail_preflight(f"SUBNET_CIDR '{raw_subnet_cidr}' must include a prefix length (e.g. '10.10.0.0/24').")
    try:
        subnet_net = ipaddress.ip_network(raw_subnet_cidr, strict=True)
    except ValueError as err:
        fail_preflight(
            f"Invalid SUBNET_CIDR '{raw_subnet_cidr}': {err}. "
            "Ensure it is a valid IPv4 CIDR with host bits set to 0."
        )
    if not isinstance(subnet_net, ipaddress.IPv4Network):
        fail_preflight(f"SUBNET_CIDR '{raw_subnet_cidr}' must be an IPv4 network, not IPv6.")
    if not is_rfc1918_network(subnet_net):
        fail_preflight(
            f"SUBNET_CIDR '{subnet_net}' is not within RFC1918 private space "
            "(10.0.0.0/8, 172.16.0.0/12, or 192.168.0.0/16)."
        )

    if PEERING_CIDR_SIZE.strip() not in ("/22", "/28"):
        fail_preflight(f"PEERING_CIDR_SIZE must be '/22' or '/28', got '{PEERING_CIDR_SIZE}'.")
    peering_prefix = int(PEERING_CIDR_SIZE.strip().lstrip("/"))
    support_prefix = 28

    # Parse base IPs and construct full network objects even if AUTO_ALLOCATE is toggled,
    # and enforce strict network boundary & overlap checks when explicit IPs are used.
    peering_net = None
    support_net = None

    if not AUTO_ALLOCATE:
        try:
            peering_ip = ipaddress.IPv4Address(PEERING_RANGE_BASE_IP.strip())
        except ValueError as err:
            fail_preflight(f"Invalid PEERING_RANGE_BASE_IP '{PEERING_RANGE_BASE_IP}': {err}")
        try:
            peering_net = ipaddress.IPv4Network(f"{peering_ip}/{peering_prefix}", strict=True)
        except ValueError as err:
            fail_preflight(
                f"PEERING_RANGE_BASE_IP '{peering_ip}' is not aligned to a {PEERING_CIDR_SIZE} boundary: {err}"
            )
        if not is_rfc1918_network(peering_net):
            fail_preflight(
                f"Runtime peering range '{peering_net}' is not within RFC1918 private space "
                "(10.0.0.0/8, 172.16.0.0/12, or 192.168.0.0/16)."
            )

        try:
            support_ip = ipaddress.IPv4Address(SUPPORT_RANGE_BASE_IP.strip())
        except ValueError as err:
            fail_preflight(f"Invalid SUPPORT_RANGE_BASE_IP '{SUPPORT_RANGE_BASE_IP}': {err}")
        try:
            support_net = ipaddress.IPv4Network(f"{support_ip}/{support_prefix}", strict=True)
        except ValueError as err:
            fail_preflight(
                f"SUPPORT_RANGE_BASE_IP '{support_ip}' is not aligned to a /28 boundary: {err}"
            )
        if not is_rfc1918_network(support_net):
            fail_preflight(
                f"Support peering range '{support_net}' is not within RFC1918 private space "
                "(10.0.0.0/8, 172.16.0.0/12, or 192.168.0.0/16)."
            )

        # 4. Verify pairwise non-overlap between SUBNET_CIDR, runtime range, and support range
        if subnet_net.overlaps(peering_net):
            fail_preflight(
                f"Overlap detected: DEPLOY subnet '{subnet_net}' overlaps RESERVE runtime range '{peering_net}'."
            )
        if subnet_net.overlaps(support_net):
            fail_preflight(
                f"Overlap detected: DEPLOY subnet '{subnet_net}' overlaps RESERVE support range '{support_net}'."
            )
        if peering_net.overlaps(support_net):
            fail_preflight(
                f"Overlap detected: RESERVE runtime range '{peering_net}' overlaps RESERVE support range '{support_net}'."
            )

    # 5. Check against existing subnets and existing global reserved ranges in the project
    existing_subnets_raw = run(
        f"gcloud compute networks subnets list --project={shlex.quote(proj_id)} --format=json",
        f"List existing regional subnetworks in project '{proj_id}'",
    )
    existing_subnets = json.loads(existing_subnets_raw) if existing_subnets_raw else []

    existing_global_addrs_raw = run(
        f"gcloud compute addresses list --global --project={shlex.quote(proj_id)} --format=json",
        f"List existing global reserved IP ranges in project '{proj_id}'",
    )
    existing_global_addrs = json.loads(existing_global_addrs_raw) if existing_global_addrs_raw else []

    candidate_ranges: list[tuple[str, str, ipaddress.IPv4Network]] = [
        (SUBNET_NAME.strip(), "DEPLOY (regional)", subnet_net),
    ]
    if peering_net is not None and support_net is not None:
        candidate_ranges.append((PEERING_RANGE_NAME.strip(), "RESERVE (global)", peering_net))
        candidate_ranges.append((SUPPORT_RANGE_NAME.strip(), "RESERVE (global)", support_net))

    # Compare against existing subnets
    for sub in existing_subnets:
        sub_name = sub.get("name", "")
        sub_region = sub.get("region", "").split("/")[-1]
        sub_network = sub.get("network", "").split("/")[-1]
        sub_cidr_str = sub.get("ipCidrRange")
        if not sub_cidr_str:
            continue
        try:
            existing_sub_net = ipaddress.IPv4Network(sub_cidr_str, strict=False)
        except ValueError:
            continue

        for cand_name, cand_type, cand_net in candidate_ranges:
            # Allow idempotent re-run if the exact target subnet already exists with the same CIDR
            if (
                cand_type == "DEPLOY (regional)"
                and sub_name == SUBNET_NAME.strip()
                and sub_region == reg
                and sub_network == NETWORK_NAME.strip()
                and existing_sub_net == cand_net
            ):
                continue
            if cand_net.overlaps(existing_sub_net):
                fail_preflight(
                    f"Planned {cand_type} '{cand_name}' ({cand_net}) overlaps existing subnet "
                    f"'{sub_name}' ({existing_sub_net}) in network '{sub_network}' (region '{sub_region}')."
                )

        for sec in sub.get("secondaryIpRanges", []) or []:
            sec_cidr = sec.get("ipCidrRange")
            sec_name = sec.get("rangeName", "secondary")
            if not sec_cidr:
                continue
            try:
                existing_sec_net = ipaddress.IPv4Network(sec_cidr, strict=False)
            except ValueError:
                continue
            for cand_name, cand_type, cand_net in candidate_ranges:
                if cand_net.overlaps(existing_sec_net):
                    fail_preflight(
                        f"Planned {cand_type} '{cand_name}' ({cand_net}) overlaps existing secondary range "
                        f"'{sec_name}' ({existing_sec_net}) on subnet '{sub_name}'."
                    )

    # Compare against existing global reserved IP ranges
    for addr in existing_global_addrs:
        addr_name = addr.get("name", "")
        addr_ip = addr.get("address")
        addr_prefix = addr.get("prefixLength")
        if not addr_ip:
            continue
        cidr_str = f"{addr_ip}/{addr_prefix}" if addr_prefix is not None else f"{addr_ip}/32"
        try:
            existing_addr_net = ipaddress.IPv4Network(cidr_str, strict=False)
        except ValueError:
            continue

        for cand_name, cand_type, cand_net in candidate_ranges:
            # Allow idempotent re-run if the exact target global range already exists with the same CIDR
            if (
                cand_type == "RESERVE (global)"
                and addr_name == cand_name
                and existing_addr_net == cand_net
            ):
                continue
            if cand_net.overlaps(existing_addr_net):
                fail_preflight(
                    f"Planned {cand_type} '{cand_name}' ({cand_net}) overlaps existing global address/range "
                    f"'{addr_name}' ({existing_addr_net})."
                )

    # Compute usable host count in GCP (total addresses minus 4 GCP-reserved addresses)
    usable_hosts = max(0, subnet_net.num_addresses - 4)
    STATE["subnet_usable_hosts"] = str(usable_hosts)

    peering_plan_cidr = (
        f"<auto-allocated>{PEERING_CIDR_SIZE}" if AUTO_ALLOCATE else str(peering_net)
    )
    support_plan_cidr = (
        "<auto-allocated>/28" if AUTO_ALLOCATE else str(support_net)
    )
    STATE["allocated_peering_cidr"] = peering_plan_cidr
    STATE["allocated_support_cidr"] = support_plan_cidr

    # 6. Render the validated ✅ Plan Table distinguishing DEPLOY (regional) vs RESERVE (global)
    plan_rows = [
        (
            SUBNET_NAME.strip(),
            "DEPLOY (regional)",
            reg,
            str(subnet_net),
            f"Customer VPC workloads ({usable_hosts} usable GCP host IPs)",
        ),
        (
            PEERING_RANGE_NAME.strip(),
            "RESERVE (global)",
            "global (no region)",
            peering_plan_cidr,
            f"Apigee runtime peering allocation ({1024 if peering_prefix == 22 else 16} total IPs)",
        ),
        (
            SUPPORT_RANGE_NAME.strip(),
            "RESERVE (global)",
            "global (no region)",
            support_plan_cidr,
            "Apigee support/troubleshooting peering allocation (16 total IPs)",
        ),
    ]

    print("\n✅ PRE-FLIGHT VALIDATION PASSED — VALIDATED NETWORKING PLAN:")
    header = f"{'Resource Name':<24} | {'Type':<18} | {'Scope / Region':<18} | {'CIDR':<22} | {'Purpose'}"
    sep = "-" * len(header)
    print(sep)
    print(header)
    print(sep)
    for r_name, r_type, r_scope, r_cidr, r_purpose in plan_rows:
        print(f"{r_name:<24} | {r_type:<18} | {r_scope:<18} | {r_cidr:<22} | {r_purpose}")
    print(sep)

    if DRY_RUN:
        print(
            "\n⚠️ DRY_RUN is currently True. Review the validated plan above and the "
            "printed gcloud commands below, then re-run with DRY_RUN=False to apply."
        )

    record_step(
        1,
        "Pre-Flight Validation",
        "COMPLETED",
        "All RFC1918, prefix, and overlap checks passed",
    )
except Exception as exc:
    if STEP_RESULTS.get(1, {}).get("status") != "FAILED":
        record_step(1, "Pre-Flight Validation", "FAILED", str(exc))
    raise

### 🔌 Step 2: Enable Required Google Cloud APIs
* **What it does:** Ensures the three Google Cloud APIs required for Apigee X VPC preparation are enabled on `PROJECT_ID`:
  * `compute.googleapis.com` (Compute Engine API — VPC networks, regional subnets, and global IP address reservations)
  * `servicenetworking.googleapis.com` (Service Networking API — private services access peering for the reserved ranges)
  * `apigee.googleapis.com` (Apigee API — prerequisite for subsequent Apigee organization provisioning)
* **Expected duration:** `~10–45 seconds` (instant if already enabled).
* **Plane:** `PROJECT PREREQUISITE` (enables control-plane APIs required before `DEPLOY` and `RESERVE` steps).

In [ ]:
#@title 🔌 Step 2: Enable Required Google Cloud APIs

REQUIRED_APIS = (
    "compute.googleapis.com",
    "servicenetworking.googleapis.com",
    "apigee.googleapis.com",
)

try:
    proj_id = PROJECT_ID.strip()
    filter_expr = " OR ".join(f"config.name={api}" for api in REQUIRED_APIS)
    enabled_out = run(
        f"gcloud services list --enabled --project={shlex.quote(proj_id)} "
        f"--filter={shlex.quote(filter_expr)} --format=json",
        f"Check already-enabled APIs in project '{proj_id}'",
    )
    enabled_items = json.loads(enabled_out) if enabled_out else []
    already_enabled = {
        item.get("config", {}).get("name", "")
        for item in enabled_items
        if isinstance(item, dict)
    }
    missing_apis = [api for api in REQUIRED_APIS if api not in already_enabled]

    if not DRY_RUN and not missing_apis:
        print(f"ℹ️ All required APIs are already enabled: {', '.join(REQUIRED_APIS)}")
        record_step(2, "Enable APIs", "SKIPPED", "already exists")
    else:
        apis_to_enable = REQUIRED_APIS if DRY_RUN else tuple(missing_apis)
        run(
            f"gcloud services enable {' '.join(apis_to_enable)} --project={shlex.quote(proj_id)}",
            f"Enable required APIs ({', '.join(apis_to_enable)})",
        )
        record_step(
            2,
            "Enable APIs",
            "COMPLETED",
            f"Enabled {', '.join(apis_to_enable)}" + (" [DRY_RUN]" if DRY_RUN else ""),
        )
except Exception as exc:
    record_step(2, "Enable APIs", "FAILED", str(exc))
    raise

### 🕸️ Step 3: VPC Network (Custom Subnet Mode)
* **What it does:** Checks whether `NETWORK_NAME` already exists in `PROJECT_ID`. If it exists, reuses it (`ℹ️`). Otherwise, when `CREATE_NETWORK_IF_MISSING` is `True`, creates the VPC network with `--subnet-mode=custom` so Google Cloud does not auto-create regional subnets in `10.128.0.0/9` that could collide with your IP plan.
* **Expected duration:** `~10–25 seconds` (or `< 3 seconds` if reusing an existing VPC).
* **Plane:** `VPC FOUNDATION` (creates/reuses the parent VPC container that hosts both the `DEPLOY (regional)` subnet in Step 4 and the `RESERVE (global)` IP allocations in Step 5).

In [ ]:
#@title 🕸️ Step 3: VPC Network

try:
    proj_id = PROJECT_ID.strip()
    net_name = NETWORK_NAME.strip()

    net_list_out = run(
        f"gcloud compute networks list --project={shlex.quote(proj_id)} "
        f"--filter={shlex.quote(f'name={net_name}')} --format=json",
        f"Check whether VPC network '{net_name}' already exists",
    )
    existing_nets = json.loads(net_list_out) if net_list_out else []

    if existing_nets:
        print(f"ℹ️ VPC network '{net_name}' already exists in project '{proj_id}'; reusing it.")
        record_step(3, "VPC Network", "SKIPPED", "already exists")
    else:
        if not CREATE_NETWORK_IF_MISSING:
            raise RuntimeError(
                f"VPC network '{net_name}' does not exist in project '{proj_id}' "
                "and CREATE_NETWORK_IF_MISSING is False."
            )
        run(
            f"gcloud compute networks create {shlex.quote(net_name)} "
            f"--project={shlex.quote(proj_id)} --subnet-mode=custom",
            f"Create custom-mode VPC network '{net_name}'",
        )
        record_step(
            3,
            "VPC Network",
            "COMPLETED",
            f"Created network '{net_name}' (--subnet-mode=custom)" + (" [DRY_RUN]" if DRY_RUN else ""),
        )
except Exception as exc:
    record_step(3, "VPC Network", "FAILED", str(exc))
    raise

### 🚀 Step 4: DEPLOY the Regional Subnetwork (`DEPLOY — Regional Resource`)
* **What it does:** **DEPLOYS** a real regional subnetwork (`SUBNET_NAME`) inside `NETWORK_NAME` in `REGION` with CIDR `SUBNET_CIDR`, optionally enabling **Private Google Access** (`--enable-private-ip-google-access`) and **VPC Flow Logs** (`--enable-flow-logs`).
  * This is the **only actual subnetwork deployed** by this notebook.
  * It hosts your customer-side workloads in `REGION` (PSC NEGs, load-balancer bridges, Envoy proxies, test VMs, and private backends).
* **Expected duration:** `~10–25 seconds`.
* **Plane:** **`DEPLOY (regional)`** — provisions a real regional subnet in your VPC.

In [ ]:
#@title 🚀 Step 4: DEPLOY the Regional Subnetwork (Real Regional Resource in Your VPC)

try:
    proj_id = PROJECT_ID.strip()
    net_name = NETWORK_NAME.strip()
    sub_name = SUBNET_NAME.strip()
    reg = REGION.strip()
    sub_cidr = SUBNET_CIDR.strip()

    subnet_net = ipaddress.IPv4Network(sub_cidr, strict=True)
    usable_hosts = max(0, subnet_net.num_addresses - 4)
    STATE["subnet_usable_hosts"] = str(usable_hosts)

    sub_list_out = run(
        f"gcloud compute networks subnets list --project={shlex.quote(proj_id)} "
        f"--regions={shlex.quote(reg)} --filter={shlex.quote(f'name={sub_name}')} --format=json",
        f"Check if regional subnetwork '{sub_name}' already exists in '{reg}'",
    )
    existing_subs = json.loads(sub_list_out) if sub_list_out else []

    if existing_subs:
        sub_obj = existing_subs[0]
        actual_cidr = sub_obj.get("ipCidrRange", sub_cidr)
        actual_net = sub_obj.get("network", "").split("/")[-1]
        if actual_net and actual_net != net_name:
            raise RuntimeError(
                f"Subnetwork '{sub_name}' in region '{reg}' already exists but belongs to "
                f"network '{actual_net}' instead of '{net_name}'."
            )
        print(f"ℹ️ Regional subnet '{sub_name}' already exists in '{reg}'.")
        print(
            f"   • Subnet Name:       {sub_name}\n"
            f"   • Plane / Action:    DEPLOY (regional)\n"
            f"   • Region:            {reg}\n"
            f"   • CIDR Range:        {actual_cidr}\n"
            f"   • Usable Host Count: {usable_hosts} (out of {subnet_net.num_addresses} total IPs; 4 reserved by GCP)"
        )
        record_step(4, "Deploy Regional Subnet", "SKIPPED", "already exists")
    else:
        create_cmd = [
            "gcloud",
            "compute",
            "networks",
            "subnets",
            "create",
            sub_name,
            f"--project={proj_id}",
            f"--network={net_name}",
            f"--region={reg}",
            f"--range={sub_cidr}",
        ]
        if ENABLE_PRIVATE_GOOGLE_ACCESS:
            create_cmd.append("--enable-private-ip-google-access")
        if ENABLE_FLOW_LOGS:
            create_cmd.append("--enable-flow-logs")

        run(
            create_cmd,
            f"DEPLOY regional subnetwork '{sub_name}' ({sub_cidr}) in region '{reg}'",
        )
        print(
            f"\n📦 Deployed Regional Subnetwork Details:\n"
            f"   • Subnet Name:       {sub_name}\n"
            f"   • Plane / Action:    DEPLOY (regional)\n"
            f"   • Region:            {reg}\n"
            f"   • CIDR Range:        {sub_cidr}\n"
            f"   • Usable Host Count: {usable_hosts} (out of {subnet_net.num_addresses} total IPs; 4 reserved by GCP)"
        )
        record_step(
            4,
            "Deploy Regional Subnet",
            "COMPLETED",
            f"name='{sub_name}', region='{reg}', cidr='{sub_cidr}', usable_hosts={usable_hosts}"
            + (" [DRY_RUN]" if DRY_RUN else ""),
        )
except Exception as exc:
    record_step(4, "Deploy Regional Subnet", "FAILED", str(exc))
    raise

Nothing is deployed here. These are IP reservations consumed later by the Apigee tenant project.

### 📌 Step 5: RESERVE the Peering IP Ranges (`RESERVE — Global Allocations Only`)
* **What it does:** Creates two **global internal IP address reservations** (`--global --purpose=VPC_PEERING`) on `NETWORK_NAME` and then runs `gcloud compute addresses describe` on each to inspect and print the exact allocated CIDR block (critical when `AUTO_ALLOCATE=True`).
* **Why two separate ranges are required by Apigee X:**
  1. **Apigee Runtime Peering Range (`PEERING_RANGE_NAME` — `/22` or `/28`):** Consumed by the Google-managed Apigee tenant project to assign internal IPs to Apigee runtime pods, ingress proxies, and internal load balancers across the Service Networking peering boundary.
  2. **Apigee Support Peering Range (`SUPPORT_RANGE_NAME` — always `/28`):** Required by Google Cloud / Apigee Service Networking for non-overlapping control-plane management, health checks, and troubleshooting inside the peered tenant project. It **must always be `/28`** regardless of whether the runtime range is `/22` or `/28`.
* **Expected duration:** `~10–25 seconds`.
* **Plane:** **`RESERVE (global, not deployed)`** — allocates global IP ranges only; no subnetworks or workloads are created in your VPC.

In [ ]:
#@title 📌 Step 5: RESERVE the Peering IP Ranges (Global Allocations Only — Nothing Deployed)


def ensure_peering_reservation(
    range_name: str,
    prefix_len: int,
    base_ip: str,
    role_label: str,
) -> tuple[bool, str]:
    """Idempotently creates a global VPC_PEERING address reservation and describes its allocated CIDR.

    Returns (created_new, allocated_cidr).
    """
    proj_id = PROJECT_ID.strip()
    net_name = NETWORK_NAME.strip()

    list_out = run(
        f"gcloud compute addresses list --global --project={shlex.quote(proj_id)} "
        f"--filter={shlex.quote(f'name={range_name}')} --format=json",
        f"Check if global peering reservation '{range_name}' ({role_label}) already exists",
    )
    existing = json.loads(list_out) if list_out else []
    created_new = False

    if existing:
        print(f"ℹ️ Global reservation '{range_name}' ({role_label}) already exists; reusing.")
    else:
        cmd = [
            "gcloud",
            "compute",
            "addresses",
            "create",
            range_name,
            "--global",
            f"--project={proj_id}",
            "--purpose=VPC_PEERING",
            f"--network={net_name}",
            f"--prefix-length={prefix_len}",
        ]
        if not AUTO_ALLOCATE:
            cmd.append(f"--addresses={base_ip}")

        run(
            cmd,
            f"RESERVE global IP range '{range_name}' ({role_label}, /{prefix_len})",
        )
        created_new = True

    desc_out = run(
        f"gcloud compute addresses describe {shlex.quote(range_name)} "
        f"--global --project={shlex.quote(proj_id)} --format=json",
        f"Describe reserved global IP range '{range_name}' to confirm allocated CIDR",
    )
    if desc_out:
        desc_obj = json.loads(desc_out)
        addr = desc_obj.get("address", base_ip if not AUTO_ALLOCATE else "<auto>")
        plen = desc_obj.get("prefixLength", prefix_len)
        allocated_cidr = f"{addr}/{plen}"
    else:
        allocated_cidr = (
            f"<auto-allocated>/{prefix_len}" if AUTO_ALLOCATE else f"{base_ip}/{prefix_len}"
        )

    print(
        f"   📌 {role_label} Reservation (GLOBAL — NOT DEPLOYED):\n"
        f"      • Name:           {range_name}\n"
        f"      • Plane / Action: RESERVE (global)\n"
        f"      • Allocated CIDR: {allocated_cidr}"
    )
    return created_new, allocated_cidr


try:
    runtime_prefix_len = int(PEERING_CIDR_SIZE.strip().lstrip("/"))
    support_prefix_len = 28

    created_runtime, runtime_cidr = ensure_peering_reservation(
        range_name=PEERING_RANGE_NAME.strip(),
        prefix_len=runtime_prefix_len,
        base_ip=PEERING_RANGE_BASE_IP.strip(),
        role_label="Apigee Runtime Range",
    )
    STATE["allocated_peering_cidr"] = runtime_cidr

    created_support, support_cidr = ensure_peering_reservation(
        range_name=SUPPORT_RANGE_NAME.strip(),
        prefix_len=support_prefix_len,
        base_ip=SUPPORT_RANGE_BASE_IP.strip(),
        role_label="Apigee Support Range",
    )
    STATE["allocated_support_cidr"] = support_cidr

    if not DRY_RUN and not created_runtime and not created_support:
        record_step(
            5,
            "Reserve Peering Ranges",
            "SKIPPED",
            f"already exists: {PEERING_RANGE_NAME.strip()}={runtime_cidr}, {SUPPORT_RANGE_NAME.strip()}={support_cidr}",
        )
    else:
        record_step(
            5,
            "Reserve Peering Ranges",
            "COMPLETED",
            f"runtime={runtime_cidr}, support={support_cidr}" + (" [DRY_RUN]" if DRY_RUN else ""),
        )
except Exception as exc:
    record_step(5, "Reserve Peering Ranges", "FAILED", str(exc))
    raise

### 🔗 Step 6: Connect Service Networking Peering (Conditional on `CONNECT_PEERING`)
* **What it does:** When `CONNECT_PEERING` is `True`, hands both global IP reservations (`PEERING_RANGE_NAME` and `SUPPORT_RANGE_NAME`) to `servicenetworking.googleapis.com`:
  * If no Service Networking peering exists yet on `NETWORK_NAME`, runs `gcloud services vpc-peerings connect --async --format=json`.
  * If a Service Networking peering **already exists** on `NETWORK_NAME`, explicitly reports this and runs `gcloud services vpc-peerings update --force --async --format=json` so the range attachment is updated idempotently without failing.
  * Polls the returned long-running operation via `gcloud services vpc-peerings operations describe` until `done: true` (**no fixed sleeps**).
  * When `CONNECT_PEERING` is `False`, skips the peering connection and leaves both ranges reserved and unattached for a later step.
* **Expected duration:** `~30–90 seconds` (or instant when `CONNECT_PEERING=False`).
* **Plane:** `RESERVE HANDOFF (Service Networking Peering)` — attaches the global IP reservations from Step 5 to Google's Service Networking producer; deploys nothing in your VPC.

In [ ]:
#@title 🔗 Step 6: Connect Service Networking Peering (Conditional)

try:
    proj_id = PROJECT_ID.strip()
    net_name = NETWORK_NAME.strip()
    p_range = PEERING_RANGE_NAME.strip()
    s_range = SUPPORT_RANGE_NAME.strip()
    ranges_arg = f"{p_range},{s_range}"

    if not CONNECT_PEERING:
        STATE["peering_state"] = "RESERVED_UNATTACHED (CONNECT_PEERING=False)"
        print(
            "ℹ️ CONNECT_PEERING is False — leaving global IP reservations "
            f"'{p_range}' and '{s_range}' unattached for a later step."
        )
        record_step(
            6,
            "Connect Service Networking Peering",
            "SKIPPED",
            "CONNECT_PEERING is False",
        )
    else:
        peerings_out = run(
            f"gcloud services vpc-peerings list "
            f"--service=servicenetworking.googleapis.com "
            f"--network={shlex.quote(net_name)} "
            f"--project={shlex.quote(proj_id)} --format=json",
            f"Check existing Service Networking peerings on VPC '{net_name}'",
        )
        existing_peerings = json.loads(peerings_out) if peerings_out else []

        if existing_peerings:
            print(
                f"ℹ️ A Service Networking peering already exists on VPC '{net_name}'. "
                "Using 'gcloud services vpc-peerings update --force' semantics instead of failing."
            )
            op_out = run(
                f"gcloud services vpc-peerings update "
                f"--service=servicenetworking.googleapis.com "
                f"--network={shlex.quote(net_name)} "
                f"--project={shlex.quote(proj_id)} "
                f"--ranges={shlex.quote(ranges_arg)} "
                f"--force --async --format=json",
                f"Update Service Networking peering on '{net_name}' with --force (--ranges={ranges_arg})",
            )
        else:
            op_out = run(
                f"gcloud services vpc-peerings connect "
                f"--service=servicenetworking.googleapis.com "
                f"--network={shlex.quote(net_name)} "
                f"--project={shlex.quote(proj_id)} "
                f"--ranges={shlex.quote(ranges_arg)} "
                f"--async --format=json",
                f"Connect Service Networking peering on '{net_name}' (--ranges={ranges_arg})",
            )

        # Poll the returned long-running operation until done (no fixed sleeps)
        if op_out:
            op_data = json.loads(op_out)
            op_name = op_data.get("name", "")
            op_done = bool(op_data.get("done", False))
            poll_attempt = 0
            max_polls = 180

            while op_name and not op_done:
                poll_attempt += 1
                if poll_attempt > max_polls:
                    raise TimeoutError(
                        f"Exceeded {max_polls} polling attempts waiting for operation '{op_name}'."
                    )
                poll_raw = run(
                    f"gcloud services vpc-peerings operations describe "
                    f"--name={shlex.quote(op_name)} "
                    f"--project={shlex.quote(proj_id)} --format=json",
                    f"Poll long-running peering operation '{op_name}' (poll #{poll_attempt})",
                )
                if not poll_raw:
                    break
                op_data = json.loads(poll_raw)
                if "error" in op_data:
                    raise RuntimeError(
                        f"Service Networking operation '{op_name}' failed: "
                        f"{json.dumps(op_data['error'])}"
                    )
                op_done = bool(op_data.get("done", False))

            print(f"✅ Long-running operation '{op_name or 'peering'}' completed.")

        STATE["peering_state"] = "ACTIVE (DRY_RUN)" if DRY_RUN else "ACTIVE"
        record_step(
            6,
            "Connect Service Networking Peering",
            "COMPLETED",
            f"service=servicenetworking.googleapis.com, ranges={ranges_arg}"
            + (" [DRY_RUN]" if DRY_RUN else ""),
        )
except Exception as exc:
    STATE["peering_state"] = "FAILED"
    record_step(6, "Connect Service Networking Peering", "FAILED", str(exc))
    raise

### 🔎 Step 7: Verify Deployed Subnet, Reserved Ranges & Service Networking Peering
* **What it does:** Runs live verification checks against the Google Cloud APIs and prints actionable remediation hints (`❌`) if any verification check fails:
  1. `gcloud compute networks subnets describe` → confirms the **DEPLOYED** regional subnet exists in `REGION`.
  2. `gcloud compute addresses list --global --filter="purpose=VPC_PEERING"` → confirms both **RESERVED** global ranges (`PEERING_RANGE_NAME` and `SUPPORT_RANGE_NAME`) exist with expected prefix lengths (`22` or `28`, and `28`).
  3. `gcloud services vpc-peerings list` (and `gcloud compute networks peerings list`) → confirms both reserved ranges are attached to `servicenetworking.googleapis.com` and the VPC peering is `ACTIVE` (when `CONNECT_PEERING=True`).
* **Expected duration:** `~5–10 seconds`.
* **Plane:** `VERIFICATION` (audits both the `DEPLOY (regional)` subnet and the `RESERVE (global)` IP allocations).

In [ ]:
#@title 🔎 Step 7: Verify Deployed Subnet, Reserved Ranges & Service Networking Peering

try:
    proj_id = PROJECT_ID.strip()
    net_name = NETWORK_NAME.strip()
    sub_name = SUBNET_NAME.strip()
    reg = REGION.strip()
    p_range = PEERING_RANGE_NAME.strip()
    s_range = SUPPORT_RANGE_NAME.strip()
    expected_runtime_prefix = int(PEERING_CIDR_SIZE.strip().lstrip("/"))
    expected_support_prefix = 28

    # 1. Verify DEPLOYED regional subnet exists in REGION
    sub_desc_out = run(
        f"gcloud compute networks subnets describe {shlex.quote(sub_name)} "
        f"--region={shlex.quote(reg)} --project={shlex.quote(proj_id)} --format=json",
        f"Verify DEPLOYED regional subnet '{sub_name}' in region '{reg}'",
    )

    # 2. Verify RESERVED global IP allocations with purpose=VPC_PEERING
    peering_addrs_out = run(
        f"gcloud compute addresses list --global --project={shlex.quote(proj_id)} "
        f"--filter={shlex.quote('purpose=VPC_PEERING')} --format=json",
        "Verify RESERVED global IP ranges (purpose=VPC_PEERING)",
    )

    # 3. Verify Service Networking peering (if CONNECT_PEERING is True)
    svc_peerings_out = ""
    net_peerings_out = ""
    if CONNECT_PEERING:
        svc_peerings_out = run(
            f"gcloud services vpc-peerings list "
            f"--service=servicenetworking.googleapis.com "
            f"--network={shlex.quote(net_name)} "
            f"--project={shlex.quote(proj_id)} --format=json",
            f"Verify Service Networking range attachment on '{net_name}'",
        )
        net_peerings_out = run(
            f"gcloud compute networks peerings list "
            f"--network={shlex.quote(net_name)} "
            f"--project={shlex.quote(proj_id)} --format=json",
            f"Verify VPC peering state on '{net_name}'",
        )

    if DRY_RUN:
        record_step(7, "Verify Networking Prerequisites", "SKIPPED", "DRY_RUN=True — verification commands printed above")
    else:
        failures: list[tuple[str, str]] = []

        # Check 1: Subnet
        if not sub_desc_out:
            failures.append((
                f"Regional subnet '{sub_name}' not found in region '{reg}'.",
                f"Remediation: Re-run Step 4 or execute: gcloud compute networks subnets create {sub_name} --network={net_name} --region={reg} --range={SUBNET_CIDR.strip()}",
            ))
        else:
            sub_data = json.loads(sub_desc_out)
            print(f"✅ Verified DEPLOYED regional subnet '{sub_name}' in '{reg}' (CIDR: {sub_data.get('ipCidrRange')}).")

        # Check 2: Global reserved ranges
        addrs_list = json.loads(peering_addrs_out) if peering_addrs_out else []
        addrs_by_name = {a.get("name"): a for a in addrs_list if isinstance(a, dict)}

        for r_name, exp_prefix in ((p_range, expected_runtime_prefix), (s_range, expected_support_prefix)):
            if r_name not in addrs_by_name:
                failures.append((
                    f"Global reserved range '{r_name}' (purpose=VPC_PEERING) is missing.",
                    f"Remediation: Re-run Step 5 or execute: gcloud compute addresses create {r_name} --global --purpose=VPC_PEERING --network={net_name} --prefix-length={exp_prefix}",
                ))
            else:
                actual_prefix = int(addrs_by_name[r_name].get("prefixLength", -1))
                actual_addr = addrs_by_name[r_name].get("address", "")
                if actual_prefix != exp_prefix:
                    failures.append((
                        f"Global reserved range '{r_name}' has prefixLength /{actual_prefix}, expected /{exp_prefix}.",
                        f"Remediation: Delete and recreate '{r_name}' with --prefix-length={exp_prefix}.",
                    ))
                else:
                    cidr_val = f"{actual_addr}/{actual_prefix}"
                    if r_name == p_range:
                        STATE["allocated_peering_cidr"] = cidr_val
                    else:
                        STATE["allocated_support_cidr"] = cidr_val
                    print(f"✅ Verified RESERVED global range '{r_name}' ({cidr_val}, purpose=VPC_PEERING).")

        # Check 3: Service Networking peering attachment & ACTIVE state
        if CONNECT_PEERING:
            svc_list = json.loads(svc_peerings_out) if svc_peerings_out else []
            attached_ranges: set[str] = set()
            for item in svc_list:
                for rng in item.get("reservedPeeringRanges", []) or []:
                    attached_ranges.add(rng)

            missing_attached = [r for r in (p_range, s_range) if r not in attached_ranges]
            if missing_attached:
                failures.append((
                    f"Ranges not attached to servicenetworking.googleapis.com: {', '.join(missing_attached)}.",
                    f"Remediation: Re-run Step 6 or execute: gcloud services vpc-peerings update --service=servicenetworking.googleapis.com --network={net_name} --ranges={p_range},{s_range} --force",
                ))
            else:
                print(f"✅ Verified both ranges ('{p_range}', '{s_range}') are attached to servicenetworking.googleapis.com.")

            net_peer_list = json.loads(net_peerings_out) if net_peerings_out else []
            sn_peering_states = [
                p.get("state", "UNKNOWN")
                for fl in net_peer_list
                for p in (fl.get("peerings", []) if isinstance(fl, dict) and "peerings" in fl else [fl])
                if isinstance(p, dict) and ("servicenetworking" in p.get("name", "") or "servicenetworking" in p.get("network", ""))
            ]
            if sn_peering_states and "ACTIVE" in sn_peering_states:
                STATE["peering_state"] = "ACTIVE"
                print("✅ Verified VPC peering state is ACTIVE.")
            elif attached_ranges:
                STATE["peering_state"] = sn_peering_states[0] if sn_peering_states else "CONNECTED"
                print(f"ℹ️ Service Networking ranges attached; VPC peering state: {STATE['peering_state']}.")
            else:
                failures.append((
                    f"No ACTIVE servicenetworking peering found on network '{net_name}'.",
                    "Remediation: Re-run Step 6 and check Service Networking API quota and permissions.",
                ))

        if failures:
            print("\n❌ VERIFICATION FAILED WITH REMEDIATION HINTS:")
            for err_msg, hint in failures:
                print(f"   ❌ {err_msg}")
                print(f"      💡 {hint}")
            raise RuntimeError("; ".join(msg for msg, _ in failures))

        record_step(7, "Verify Networking Prerequisites", "COMPLETED", "Subnet, reservations, and peering verified")
except Exception as exc:
    record_step(7, "Verify Networking Prerequisites", "FAILED", str(exc))
    raise

### 📋 Step 8: Summary & Handoff to Apigee Provisioning Notebook
* **What it does:**
  1. Renders a boxed summary split into two clearly labelled sections:
     * **`DEPLOYED (regional)`** → VPC network, regional subnet, region, CIDR, Private Google Access, and VPC Flow Logs.
     * **`RESERVED (global, not deployed)`** → runtime range name & allocated CIDR, support range name & allocated CIDR, and Service Networking peering state.
  2. Renders the final **per-step execution status table** across Steps 0–8.
  3. Prints the exact handoff variables to paste into your Apigee X organization provisioning notebook, plus the direct **Google Cloud Console VPC URL**.
* **Expected duration:** `< 1 second`.
* **Plane:** `SUMMARY & HANDOFF` (read-only reporting).

In [ ]:
#@title 📋 Step 8: Summary & Handoff

try:
    proj_id = PROJECT_ID.strip() or "<unset-project-id>"
    reg = REGION.strip()
    net_name = NETWORK_NAME.strip()
    sub_name = SUBNET_NAME.strip()
    sub_cidr = SUBNET_CIDR.strip()
    p_range = PEERING_RANGE_NAME.strip()
    s_range = SUPPORT_RANGE_NAME.strip()
    p_cidr = STATE.get("allocated_peering_cidr", f"{PEERING_RANGE_BASE_IP.strip()}{PEERING_CIDR_SIZE.strip()}")
    s_cidr = STATE.get("allocated_support_cidr", f"{SUPPORT_RANGE_BASE_IP.strip()}/28")
    peer_state = STATE.get("peering_state", "DRY_RUN" if DRY_RUN else ("ACTIVE" if CONNECT_PEERING else "NOT_CONNECTED"))
    usable_hosts = STATE.get("subnet_usable_hosts", "252")

    record_step(8, "Summary & Handoff", "COMPLETED", "Rendered summary and handoff block")

    width = 78
    top_border = "╔" + ("═" * width) + "╗"
    mid_border = "╠" + ("═" * width) + "╣"
    sub_border = "╟" + ("─" * width) + "╢"
    bot_border = "╚" + ("═" * width) + "╝"

    def box_line(text: str = "") -> str:
        wide_chars = sum(1 for ch in text if ord(ch) > 0x2600)
        pad_width = max(0, (width - 2) - wide_chars)
        return f"║ {text:<{pad_width}} ║"

    box_lines = [
        top_border,
        box_line("🌐 APIGEE X — VPC NETWORKING PREREQUISITES SUMMARY"),
        box_line(f"Project ID: {proj_id}   |   Mode: {'DRY_RUN (Nothing Executed)' if DRY_RUN else 'LIVE EXECUTION'}"),
        mid_border,
        box_line("🚀 SECTION 1: DEPLOYED (regional) — Real Customer VPC Resource"),
        sub_border,
        box_line(f"VPC Network            : {net_name} (custom subnet mode)"),
        box_line(f"Regional Subnetwork    : {sub_name}"),
        box_line(f"Region                 : {reg}"),
        box_line(f"Subnet CIDR            : {sub_cidr} ({usable_hosts} usable GCP host IPs)"),
        box_line(f"Private Google Access  : {'Enabled' if ENABLE_PRIVATE_GOOGLE_ACCESS else 'Disabled'}"),
        box_line(f"VPC Flow Logs          : {'Enabled' if ENABLE_FLOW_LOGS else 'Disabled'}"),
        mid_border,
        box_line("📌 SECTION 2: RESERVED (global, not deployed) — Apigee Tenant Allocations"),
        sub_border,
        box_line(f"Runtime Range Name     : {p_range}"),
        box_line(f"Runtime Allocated CIDR : {p_cidr} (global VPC_PEERING allocation)"),
        box_line(f"Support Range Name     : {s_range}"),
        box_line(f"Support Allocated CIDR : {s_cidr} (global VPC_PEERING allocation)"),
        box_line(f"Peering State          : {peer_state}"),
        bot_border,
    ]
    print("\n" + "\n".join(box_lines))

    # Render Final Per-Step Status Table
    print("\n📊 PER-STEP EXECUTION STATUS TABLE:")
    tbl_header = f"{'Step':<8} | {'Step Name':<36} | {'Status':<10} | {'Details'}"
    tbl_sep = "-" * len(tbl_header)
    print(tbl_sep)
    print(tbl_header)
    print(tbl_sep)
    for s_num in sorted(STEP_RESULTS.keys()):
        entry = STEP_RESULTS[s_num]
        st = entry["status"]
        badge = "✅" if st == "COMPLETED" else ("ℹ️" if st == "SKIPPED" else "❌")
        print(
            f"{entry['step']:<8} | {entry['name']:<36} | {badge} {st:<7} | {entry['detail']}"
        )
    print(tbl_sep)

    # Handoff block for the Apigee X Organization Provisioning Notebook
    vpc_console_url = (
        f"https://console.cloud.google.com/networking/networks/details/{net_name}?project={proj_id}"
    )
    print("\n📋 HANDOFF — COPY & PASTE INTO YOUR APIGEE X PROVISIONING NOTEBOOK:")
    print("```python")
    print(f'PROJECT_ID = "{proj_id}"')
    print(f'REGION = "{reg}"')
    print(f'NETWORK_NAME = "{net_name}"')
    print(f'AUTHORIZED_NETWORK = "projects/{proj_id}/global/networks/{net_name}"')
    print(f'SUBNET_NAME = "{sub_name}"  # DEPLOYED (regional) in {reg}')
    print(f'SUBNET_CIDR = "{sub_cidr}"')
    print(f'PEERING_RANGE_NAME = "{p_range}"  # RESERVED (global, not deployed)')
    print(f'PEERING_RANGE_CIDR = "{p_cidr}"')
    print(f'SUPPORT_RANGE_NAME = "{s_range}"  # RESERVED (global, not deployed)')
    print(f'SUPPORT_RANGE_CIDR = "{s_cidr}"')
    print("```")
    print(f"\n🔗 Cloud Console VPC URL: {vpc_console_url}")
except Exception as exc:
    record_step(8, "Summary & Handoff", "FAILED", str(exc))
    raise

### 🧹 Appendix: Teardown Procedure (Commented Out for Safety)
* **What it does:** Provides the exact reverse-dependency teardown commands to cleanly remove the resources prepared by this notebook (only after any Apigee X organization using them has been deleted).
* **Mandatory Teardown Order:**
  1. **Delete Service Networking peering** (`gcloud services vpc-peerings delete`)
  2. **Release global IP reservations** (`gcloud compute addresses delete --global`)
  3. **Delete regional subnetwork** (`gcloud compute networks subnets delete --region=...`)
  4. **Delete VPC network** (`gcloud compute networks delete`)
* **Expected duration:** `~1–2 minutes` (only if manually uncommented and executed).
* **Plane:** `TEARDOWN (commented out — safe during Run All)`.

In [ ]:
#@title 🧹 Appendix: Teardown Commands (Commented Out — Un-comment Only to Destroy Resources)

# ⚠️ WARNING: Do NOT run teardown while an Apigee X organization is still active!
# Follow this exact dependency order:
#   1. delete peering -> 2. release reservations -> 3. delete subnet -> 4. delete network

# --- 1. Delete Service Networking VPC Peering ---
# run(
#     f"gcloud services vpc-peerings delete "
#     f"--service=servicenetworking.googleapis.com "
#     f"--network={shlex.quote(NETWORK_NAME.strip())} "
#     f"--project={shlex.quote(PROJECT_ID.strip())} --quiet",
#     "1/4 Delete Service Networking VPC peering",
# )

# --- 2. Release Global Reserved Peering IP Ranges (RESERVED plane) ---
# run(
#     f"gcloud compute addresses delete "
#     f"{shlex.quote(PEERING_RANGE_NAME.strip())} {shlex.quote(SUPPORT_RANGE_NAME.strip())} "
#     f"--global --project={shlex.quote(PROJECT_ID.strip())} --quiet",
#     "2/4 Release global VPC_PEERING IP reservations",
# )

# --- 3. Delete Deployed Regional Subnetwork (DEPLOYED plane) ---
# run(
#     f"gcloud compute networks subnets delete {shlex.quote(SUBNET_NAME.strip())} "
#     f"--region={shlex.quote(REGION.strip())} "
#     f"--project={shlex.quote(PROJECT_ID.strip())} --quiet",
#     "3/4 Delete deployed regional subnetwork",
# )

# --- 4. Delete Custom-Mode VPC Network ---
# run(
#     f"gcloud compute networks delete {shlex.quote(NETWORK_NAME.strip())} "
#     f"--project={shlex.quote(PROJECT_ID.strip())} --quiet",
#     "4/4 Delete VPC network",
# )

print("ℹ️ Teardown appendix is safely commented out. Uncomment lines above only when tearing down.")